In [1]:
from __future__ import division, print_function
import os
import sys
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.models import load_model
import tensorflow as tf

In [2]:
def load_model_wrapper(model_h5):
    # read .h5 model
    custom_objects={"tf": tf}    
    get_custom_objects().update(custom_objects)    
    model=load_model(model_h5)
    #print("got the model")
    #model.summary()
    return model

In [3]:
model_h5="/mnt/lab_data2/anusri/chrombpnet/results/chrombpnet/ATAC_PE/GM12878/nautilus_runs/GM12878_03.01.2022_bias_128_4_1234_0.4_fold_0/chrombpnet_model//chrombpnet_wo_bias.h5"
model = load_model_wrapper(model_h5)

In [4]:
import pandas as pd 

In [5]:
NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

nonpeaks = pd.read_csv("/mnt/lab_data2/anusri/chrombpnet/results/chrombpnet/ATAC_PE/GM12878/nautilus_runs/GM12878_03.01.2022_bias_128_4_1234_0.4_fold_0/chrombpnet_model//filtered.nonpeaks.bed", sep="\t", header=None,  names=NARROWPEAK_SCHEMA)



In [6]:
nonpeaks.head()

,chr,start,end,1,2,3,4,5,6,summit
0,chr19,53808000,53810114,.,.,.,.,.,.,1057
1,chr9,90208000,90210114,.,.,.,.,.,.,1057
2,chr10,56067000,56069114,.,.,.,.,.,.,1057
3,chr11,3137000,3139114,.,.,.,.,.,.,1057
4,chr22,42212000,42214114,.,.,.,.,.,.,1057


In [7]:
import json
chr_fold_path="/mnt/lab_data2/anusri/chrombpnet/splits/fold_0.json"
splits_dict = json.load(open(chr_fold_path))
chroms_to_keep = set(splits_dict["test"])
regions_subsample = nonpeaks[(nonpeaks["chr"].isin(chroms_to_keep))]

In [74]:
import utils
def get_seq_shuffled(peaks_df, genome, width):
    """
    Same as get_cts, but fetches sequence from a given genome.
    """
    vals = []

    for i, r in peaks_df.iterrows():
        sequence = str(genome[r['chr']][(r['start']+r['summit'] - width//2):(r['start'] + r['summit'] + width//2)])
        vals.extend(utils.dinuc_shuffle(sequence, num_shufs=10))

    return utils.dna_to_one_hot(vals)

def get_seq(peaks_df, genome, width):
    """
    Same as get_cts, but fetches sequence from a given genome.
    """
    vals = []

    for i, r in peaks_df.iterrows():
        sequence = str(genome[r['chr']][(r['start']+r['summit'] - width//2):(r['start'] + r['summit'] + width//2)])
        vals.append(sequence)

    return utils.dna_to_one_hot(vals)

In [9]:
import pyfaidx
import numpy as np
inputlen = model.input_shape[1] 
genome_fasta = pyfaidx.Fasta("/mnt/lab_data2/anusri/chrombpnet/reference/hg38.genome.fa")
seqs = get_seq(regions_subsample, genome_fasta, inputlen)

In [31]:
def softmax(x, temp=1):
    norm_x = x - np.mean(x,axis=1, keepdims=True)
    return np.exp(temp*norm_x)/np.sum(np.exp(temp*norm_x), axis=1, keepdims=True)

index=np.random.randint(0, seqs.shape[0], size=1000)
pred_output=model.predict(seqs[index], batch_size=64, verbose=True)
footprint_for_motif_fwd = softmax(pred_output[0])*(np.exp(pred_output[1])-1)

w_mot_seqs_revc = seqs[index][:, ::-1, ::-1]
pred_output_rev=model.predict(w_mot_seqs_revc, batch_size=64, verbose=True)
footprint_for_motif_rev = softmax(pred_output_rev[0])*(np.exp(pred_output_rev[1])-1)

16/16 [==============================] - 6s 381ms/step


In [66]:
 #plt.hist(pred_output_rev[1], bins=100)

In [76]:
seqs_shuff = get_seq_shuffled(regions_subsample.sample(1000), genome_fasta, inputlen)

In [77]:
#index=np.random.randint(0, seqs.shape[0], size=1000)
pred_output=model.predict(seqs_shuff, batch_size=64, verbose=True)
footprint_for_motif_fwd = softmax(pred_output[0])*(np.exp(pred_output[1])-1)

w_mot_seqs_revc = seqs_shuff[:, ::-1, ::-1]
pred_output_rev=model.predict(w_mot_seqs_revc, batch_size=64, verbose=True)
footprint_for_motif_rev = softmax(pred_output_rev[0])*(np.exp(pred_output_rev[1])-1)

157/157 [==============================] - 58s 370ms/step


In [78]:
np.mean(pred_output[1])

5.821169

In [79]:
np.mean(pred_output_rev[1])

5.815649

In [22]:
import matplotlib.pyplot as plt

In [34]:
np.mean(pred_output[1])

5.6055098

In [35]:
np.mean(pred_output_rev[1])

5.6051087

In [52]:
counts_for_motif = (np.exp(pred_output_rev[1]) + np.exp(pred_output[1]))/2

In [53]:
counts_for_motif.mean(0)

array([324.67694], dtype=float32)

In [57]:
np.log(counts_for_motif.mean(0))-1

array([4.475978], dtype=float32)

In [64]:
counts_for_motif = (np.exp(pred_output_rev[1]) + np.exp(pred_output[1]) - 2)/2
np.log(counts_for_motif.mean(0))

array([5.779746], dtype=float32)

In [42]:
pred_output_rev[1].shape

(1000, 1)

In [65]:
np.mean(pred_output[1])

5.6055098

In [27]:
np.median(pred_output[1])

5.557767